# 05 · Ticket clock and fields

This notebook replicates the Ticket clock and fields paragraph of the report's Results section.

> Ticket clock and fields: Opening times never step backwards in case-number order (0 of 7,470 steps) but would 1,295–1,312 times on hospital-local clocks, so case times use one U.S. Eastern clock, as IPA confirmed. Trouble Code and Category are filled on 1 of 64 open tickets and on nearly all of the 461 August 2026 tickets of any kind with status Trouble Closed. Category is set near close, and the export holds only each field's final value.

**How it works.** `case_order.py` counts the steps between consecutive tickets in case-number order and the steps that go backwards on the export clock and on hospital-local clocks. `timestamp_semantics.py` counts the tickets by status, opening month, and age at export, and it measures how often each field is filled. Both read IPA's case export. We recompute the paragraph's numbers from four of their tables, `case_order_steps.csv`, `ts_facts.csv`, `ts_field_fill_rates.csv`, and `ts_text_markers_by_month.csv`, and check them against the report. IPA's answers about the clock and the fields are records of a meeting, so the check has no rows for them. `docs/04_timestamp_semantics.md` (§2.1 and §5) describes the analysis.

**Re-run.** With `RERUN = True`, we re-run `verification/timestamp_semantics.py`, which takes about 3 minutes, and then `verification/case_order.py`, which takes about 3 s. The first rewrites its 42 `ts_*.csv` tables at the top of a scratch copy of `verification/outputs/`. The second checks its tickets by state and time zone against the `ts_zone_mapping.csv` that the first has just written, and then it rewrites `case_order_steps.csv`. We then compare each of the 43 files with the committed one. With `RERUN = False`, we skip the re-run and read the committed outputs.

**Used as committed, without re-running.** The re-run does not re-create four of the files we read, so they come from the committed outputs either way:
- We read `fu/fu_ticket_last10.csv`, which `db_followup.py` wrote from the clearView telemetry database. The notebooks do not re-read that database, so no notebook re-creates the table. It has one row per ticket and telemetry window, 24,452 rows for the 6,113 tickets matched to a clearView device. We use it for cross-checks and for the two fields of the one August 2026 Trouble Closed ticket that was opened 30 days or more before the export, so with `RERUN = True` that ticket's fields still come from the committed table.
- For cross-checks only, we read `audit_case_status_by_month.csv`, `audit_case_service_group_by_status.csv`, and `audit_case_missing_category_composition.csv` from `audit_case_logic.py`. We do not re-run that script because it also reads a text dump of the midterm notebook from the gitignored `data/derived/` folder, and a test re-run without that file stopped there with an error.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'timestamp_semantics.py')
    R.rerun(OUT, 'case_order.py')
R.compare(OUT)

timestamp_semantics.py: exit status 0, 174 s
    Done.


case_order.py: exit status 0, 3 s
    6/6 checks pass
43 files written: 43 identical.


,file written by the re-run,compared with the committed file
0,case_order_steps.csv,identical
1,ts_anchor_correction_distribution.csv,identical
2,ts_clock_hypothesis_scorecard.csv,identical
3,ts_close_duration_bins.csv,identical
4,ts_close_duration_quantiles.csv,identical
5,ts_close_hour_by_zone.csv,identical
6,ts_desk_edge_by_month.csv,identical
7,ts_dispatch_by_record_type.csv,identical
8,ts_dispatch_vs_outcome.csv,identical
9,ts_dst_desk_edge.csv,identical


## Steps in case-number order

Salesforce numbers cases in creation order, and the pseudonyms T00001 to T07471 follow the sorted 8-digit case numbers (`common.py`), so the steps are the pairs of consecutive tickets in that order, one fewer than the tickets. We count the tickets by opening month in `ts_text_markers_by_month.csv` and cross-check the total with the rows `timestamp_semantics.py` loads (`ts_facts.csv`), with the open and closed groups of `ts_field_fill_rates.csv`, and with `audit_case_logic.py`'s committed count of tickets by service group.

The steps run over all 7,471 tickets, of any kind, as in docs/04 §2.1. The report's Methods make unplanned tickets the default for ticket counts, so we also count the unplanned tickets, which are all tickets except planned work, canceled requests, and duplicates (Definitions), and the steps between them. If no step goes backwards on the export clock over all the tickets, none does among the unplanned ones either, since leaving tickets out of a sequence of times that never decreases cannot make it decrease.

In [3]:
facts = R.read_csv(OUT, 'ts_facts.csv')
fact = {(section, item): value for section, item, value in facts[['section', 'item', 'value']].itertuples(index=False)}
fill = R.read_csv(OUT, 'ts_field_fill_rates.csv').set_index('group')
tickets_by_month = R.read_csv(OUT, 'ts_text_markers_by_month.csv').set_index('month')['tickets']
service_groups = R.read_csv(OUT, 'audit_case_service_group_by_status.csv').set_index('service_group')
NOT_UNPLANNED = ['Planned work', 'Cancelled request', 'Duplicate case']  # the three kinds the Definitions leave out

n_tickets = int(tickets_by_month.sum())
assert n_tickets == int(fact['load', 'rows'])
assert n_tickets == fill.loc['open (Status=Dispatch)', 'tickets'] + fill.loc['closed (all other statuses)', 'tickets']
assert n_tickets == service_groups.loc['All', 'All'] == service_groups['All'].drop('All').sum()
n_steps = n_tickets - 1
n_unplanned = n_tickets - int(service_groups.loc[NOT_UNPLANNED, 'All'].sum())
first_month, last_month = (pd.Period(m, freq='M').strftime('%B %Y') for m in tickets_by_month.index[[0, -1]])
print(f'{n_tickets:,} tickets opened from {first_month} to {last_month} make {n_steps:,} steps between consecutive '
      f'tickets in case-number order. The {n_unplanned:,} unplanned tickets among them make {n_unplanned - 1:,}.')

7,471 tickets opened from August 2025 to August 2026 make 7,470 steps between consecutive tickets in case-number order. The 7,186 unplanned tickets among them make 7,185.


## Backward steps: the creation-order test

`case_order.py` sorts the 7,471 tickets by case number and counts the steps between consecutive tickets that go backwards in time. On one shared clock, opening times can tie, since they are minute-precise, but cannot go backwards. For hospital-local clocks, the script shows each recorded time as the hospital's own zone would, so a Pacific ticket opened minutes after an Eastern one shows about 3 hours earlier. It also works the other way around, reading each recorded time as the hospital's local time and moving it to U.S. Eastern. It does both under two maps from hospital location to time zone: `timestamp_semantics.py`'s (the state, with 3-digit ZIP rules for the states that span two zones, in `verification/zones.py`) and the state alone. Before counting, it checks that its tickets by state and zone equal `timestamp_semantics.py`'s `ts_zone_mapping.csv`. We check the report's range against the shown-locally counts on the two maps.

In [4]:
order = R.read_csv(OUT, 'case_order_steps.csv')
export, local = order.iloc[0], order.iloc[1:]
assert export.clock == 'export clock, as recorded' and export.steps == n_steps and (local.steps == n_steps).all()
shown = local[local.clock.str.startswith('shown locally')]  # each recorded time as its hospital's zone would show it
read_local = local[local.clock.str.startswith('read as local')]  # each recorded time read as local, moved to Eastern
assert len(shown) == len(read_local) == 2 and len(shown) + len(read_local) == len(local)
primary = shown[shown.zone_map.str.startswith('state + ZIP3')].iloc[0]  # timestamp_semantics.py's map
state_only = shown[shown.zone_map.str.startswith('state only')].iloc[0]  # the state alone
assert (local.backward > 1200).all()  # docs/04 §2.1: more than 1,200 backward steps on hospital-local clocks
print(f'Export clock: {export.backward} of {export.steps:,} steps go backwards, and {export.ties} stay in the same '
      f'minute.')
print(f'Shown locally: {R.span(shown.backward.min(), shown.backward.max())} backward steps over the two maps, '
      f"{primary.backward:,} on timestamp_semantics.py's map and {state_only.backward:,} on the state alone. "
      f'Read as local: {R.span(read_local.backward.min(), read_local.backward.max())}.')
order

Export clock: 0 of 7,470 steps go backwards, and 173 stay in the same minute.
Shown locally: 1,295–1,312 backward steps over the two maps, 1,312 on timestamp_semantics.py's map and 1,295 on the state alone. Read as local: 1,210–1,220.


,clock,zone_map,steps,backward,ties
0,"export clock, as recorded",NaN,7470,0,173
1,shown locally: each recorded time as its hospital's zone would show it,"state + ZIP3: the state, with 3-digit ZIP rules for split states (timestamp_semantics.py)",7470,1312,133
2,shown locally: each recorded time as its hospital's zone would show it,state only: the state alone,7470,1295,134
3,"read as local: each recorded time read as its hospital's local time, moved to US Eastern","state + ZIP3: the state, with 3-digit ZIP rules for split states (timestamp_semantics.py)",7470,1220,127
4,"read as local: each recorded time read as its hospital's local time, moved to US Eastern",state only: the state alone,7470,1210,128


**The range in docs/04.** In §2.1, docs/04 records "more than 1,200 backward steps (1,205–1,312 depending on the zone map)", and `case_order.py`'s docstring adds that the 1,205 came from a second-wave reviewer's own zone map, which is not in the repository. The read-as-local counts, 1,210 and 1,220, come from a different simulation, which docs/04 does not describe.

## One U.S. Eastern clock

That the shared clock is U.S. Eastern time is IPA's answer at the Oct 13th meeting, which docs/04 records in its update of Oct 14th and in question 1 of §6. It is a record of IPA's answer, so the check has no row for it.

## Open tickets with a Trouble Code and a Category

An open ticket has status Dispatch at export, and `ts_field_fill_rates.csv` gives the open tickets' count and the share of them that carry each field; the shares have four decimals, which pins down the count in any group under 5,000 tickets. We cross-check with `timestamp_semantics.py`'s own count of open tickets with a Trouble Code (`ts_facts.csv`), with `audit_case_logic.py`'s committed counts of open tickets missing each field, and with the open tickets among the device-matched tickets in `fu/fu_ticket_last10.csv`, which are a subset of the open tickets. The open tickets without a Category all lack a Trouble Code too, so when the two counts are equal, the tickets with a Trouble Code are the ones with a Category.

The open tickets fall in two of `audit_case_logic.py`'s service groups, Missing trouble code and Other recorded issue. Neither group is one of the three kinds that the Definitions leave out of unplanned tickets (planned work, canceled requests, and duplicates), so all 64 open tickets are unplanned, and their counts are the same under the Methods' rule that ticket counts are unplanned tickets unless stated otherwise.

In [5]:
last10 = (R.read_csv(OUT, 'fu/fu_ticket_last10.csv',
                     usecols=['case_id', 'opened_at_eastern', 'status', 'trouble_code', 'category'])
          .drop_duplicates('case_id'))  # the table has one row per ticket and window


def filled(group, field):
    """Tickets in a group of ts_field_fill_rates.csv that carry the field: the group's share times its tickets."""
    count = fill.loc[group, field] * fill.loc[group, 'tickets']
    assert abs(count - round(count)) < 0.25, (group, field, count)
    return int(round(count))


OPEN = 'open (Status=Dispatch)'
n_open = int(fill.loc[OPEN, 'tickets'])
open_code, open_category = filled(OPEN, 'Trouble Code'), filled(OPEN, 'Trouble Code Category')

# Cross-checks: timestamp_semantics.py's own count, audit_case_logic.py's committed tables, device-matched tickets
assert open_code == int(fact['C', 'open_tickets_with_trouble_code'])
groups = service_groups['Dispatch']  # open tickets by service group
assert groups.drop('All').sum() == groups['All']  # the groups partition the tickets
# The only groups of open tickets, both unplanned, so all of them are unplanned (the report's default)
assert groups['Missing trouble code'] + groups['Other recorded issue'] == groups['All']
no_category = R.read_csv(OUT, 'audit_case_missing_category_composition.csv').query("status == 'Dispatch'")
assert set(no_category.trouble_code) == {'[Missing]'}  # open tickets without a Category also lack a Trouble Code
matched_open = last10[last10.status == 'Dispatch']
open_table = pd.DataFrame(
    [[n_open, open_code, open_category],
     [groups['All'], groups['Other recorded issue'], groups['All'] - no_category.cases.sum()],
     [len(matched_open), matched_open.trouble_code.notna().sum(), matched_open.category.notna().sum()]],
    index=['ts_field_fill_rates.csv', 'audit_case_*.csv (committed)', 'fu/fu_ticket_last10.csv (device-matched)'],
    columns=['open tickets', 'with a Trouble Code', 'with a Category']).astype(int)
assert open_table.iloc[0].equals(open_table.iloc[1])
assert (open_table.iloc[2] <= open_table.iloc[0]).all()  # the device-matched open tickets are among them
# The tickets without a Category lack a Trouble Code, so with equal counts, both fields are on the same tickets
assert open_category == open_code
print(f'Of the {n_open} open tickets, all unplanned, {open_code} carries both a Trouble Code and a Category, and the '
      f'other {n_open - open_code} carry neither.')
open_table

Of the 64 open tickets, all unplanned, 1 carries both a Trouble Code and a Category, and the other 63 carry neither.


,open tickets,with a Trouble Code,with a Category
ts_field_fill_rates.csv,64,1,1
audit_case_*.csv (committed),64,1,1
fu/fu_ticket_last10.csv (device-matched),56,1,1


## The comparison month

`ts_facts.csv` lists the open tickets by opening month, and all 64 were opened in August 2026, so we compare them with the tickets opened that month with status Trouble Closed, as the report does. We leave out the month's tickets with the other closed statuses (Ricoh-Closed, Closed, and Case Closed).

In [6]:
import ast

open_by_month = ast.literal_eval(fact['C', 'open_tickets_by_month'])
assert sum(open_by_month.values()) == n_open and len(open_by_month) == 1
(month,) = open_by_month
month_label = pd.Period(month, freq='M').strftime('%B %Y')
print(f'All {n_open} open tickets were opened in {month_label}.')

All 64 open tickets were opened in August 2026.


## August 2026 tickets with status Trouble Closed

We take the tickets opened in August 2026 from `ts_text_markers_by_month.csv` and subtract those with the other four statuses, which `ts_facts.csv` lists by opening month (Dispatch, Ricoh-Closed, Closed, and Case Closed). "Of any kind" means that every record type and service group counts, planned work included. We cross-check the counts with `audit_case_logic.py`'s committed table of statuses by opening month.

In [7]:
status_counts = ast.literal_eval(fact['B', 'status_counts'])
other_statuses = {'Dispatch': open_by_month, **ast.literal_eval(fact['C', 'other_closed_statuses_by_month'])}
assert set(other_statuses) | {'Trouble Closed'} == set(status_counts)
assert all(sum(per_month.values()) == status_counts[s] for s, per_month in other_statuses.items())
in_month = {s: per_month.get(month, 0) for s, per_month in other_statuses.items()}
in_month = {'Trouble Closed': int(tickets_by_month[month]) - sum(in_month.values()), **in_month}
n_aug_closed = in_month['Trouble Closed']
n_other_closed = sum(in_month.values()) - n_aug_closed - in_month['Dispatch']

audit_month = R.read_csv(OUT, 'audit_case_status_by_month.csv').set_index('month').loc[month]
status_table = pd.DataFrame({'ts_text_markers_by_month.csv, ts_facts.csv': pd.Series(in_month),
                             'audit_case_status_by_month.csv (committed)': audit_month}).reindex(list(in_month))
status_table = status_table.astype(int)
status_table.loc['all statuses'] = status_table.sum()
assert (status_table.iloc[:, 0] == status_table.iloc[:, 1]).all()
print(f'Of the {tickets_by_month[month]} tickets opened in {month_label}, {n_aug_closed} have status Trouble Closed, '
      f"{in_month['Dispatch']} are open, and {n_other_closed} have one of the other closed statuses.")
status_table

Of the 541 tickets opened in August 2026, 461 have status Trouble Closed, 64 are open, and 16 have one of the other closed statuses.


,"ts_text_markers_by_month.csv, ts_facts.csv",audit_case_status_by_month.csv (committed)
Trouble Closed,461,461
Dispatch,64,64
Ricoh-Closed,11,11
Closed,1,1
Case Closed,4,4
all statuses,541,541


## Trouble Code and Category on the 461 ("nearly all")

`ts_field_fill_rates.csv` gives fill shares for Trouble Closed tickets by age at export, counted back from the latest timestamp in the export (Sep 1st 08:05), and its under-7-day and 7-to-30-day groups hold 460 of the 461, all opened in August; we turn their shares into counts as above. The other ticket was opened on Aug 1st, and we read its two fields in `fu/fu_ticket_last10.csv`. As a cross-check, we assert that the device-matched tickets in that table, a subset of each age group, have no more tickets or fields than the group.

In [8]:
def day(ts):
    """A date as 'Aug 2nd'."""
    suffix = 'th' if 11 <= ts.day <= 13 else {1: 'st', 2: 'nd', 3: 'rd'}.get(ts.day % 10, 'th')
    return f'{ts:%b} {ts.day}{suffix}'


def when(ts):
    """A timestamp as 'Aug 2nd 08:05'."""
    return f'{day(ts)} {ts:%H:%M}'


export_ref = pd.Timestamp(fact['load', 'latest_timestamp_any_column (export ran at/after)'])
cut_7, cut_30 = export_ref - pd.Timedelta(days=7), export_ref - pd.Timedelta(days=30)
# Every ticket aged under 30 days at export was opened in the comparison month
assert cut_30.strftime('%Y-%m') == month == pd.Timestamp(fact['load', 'last_opened']).strftime('%Y-%m')
age_groups = [(f'after {when(cut_7)} (under 7 days)', 'age <7d Trouble Closed only'),
              (f'after {when(cut_30)}, up to {when(cut_7)} (7 to 30 days)', 'age 7-30d Trouble Closed only')]
rows = [(label, int(fill.loc[group, 'tickets']), filled(group, 'Trouble Code'), filled(group, 'Trouble Code Category'))
        for label, group in age_groups]

# The rest were opened earlier in the month (30 days or more before the export); the per-ticket table holds them all
opened = pd.to_datetime(last10.opened_at_eastern)
in_comparison = (last10.status == 'Trouble Closed') & (opened.dt.strftime('%Y-%m') == month)
older = last10[in_comparison & (opened <= cut_30)]
assert len(older) == n_aug_closed - sum(row[1] for row in rows)
first_day = pd.Period(month, freq='M').start_time
rows.append((f'{day(first_day)} up to {when(cut_30)} (30 days or more)', len(older),
             int(older.trouble_code.notna().sum()), int(older.category.notna().sum())))
nearly = pd.DataFrame(rows, columns=['Trouble Closed tickets opened', 'tickets', 'with a Trouble Code',
                                     'with a Category']).set_index('Trouble Closed tickets opened')

# Cross-check: in each age group, the device-matched tickets of fu/fu_ticket_last10.csv are a subset of the group
matched_groups = [last10[in_comparison & (opened > cut_7)],
                  last10[in_comparison & (opened > cut_30) & (opened <= cut_7)], older]
matched = pd.DataFrame([(len(s), s.trouble_code.notna().sum(), s.category.notna().sum()) for s in matched_groups],
                       index=nearly.index, columns=nearly.columns)
assert (matched <= nearly).all().all()

nearly.loc[f'all of {month_label}'] = nearly.sum()
aug_code = int(nearly.loc[f'all of {month_label}', 'with a Trouble Code'])
aug_category = int(nearly.loc[f'all of {month_label}', 'with a Category'])
aug_category_share = aug_category / n_aug_closed
# Tickets with both fields: at least this many, and exactly this many when every ticket has a Trouble Code
aug_both = aug_code + aug_category - n_aug_closed
print(f'{aug_code} of the {n_aug_closed} have a Trouble Code and {aug_category} ({aug_category_share:.1%}) a Category, '
      f"so {'' if aug_code == n_aug_closed else 'at least '}{aug_both} ({aug_both / n_aug_closed:.1%}) have both. "
      f'The {matched.tickets.sum()} device-matched ones in fu/fu_ticket_last10.csv include '
      f"{matched['with a Trouble Code'].sum()} with a Trouble Code and {matched['with a Category'].sum()} with a "
      f'Category.')
nearly

461 of the 461 have a Trouble Code and 458 (99.3%) a Category, so 458 (99.3%) have both. The 401 device-matched ones in fu/fu_ticket_last10.csv include 401 with a Trouble Code and 398 with a Category.


,tickets,with a Trouble Code,with a Category
Trouble Closed tickets opened,,,
after Aug 25th 08:05 (under 7 days),51,51,51
"after Aug 2nd 08:05, up to Aug 25th 08:05 (7 to 30 days)",409,409,406
Aug 1st up to Aug 2nd 08:05 (30 days or more),1,1,1
all of August 2026,461,461,458


**"Nearly all" and docs/04's counts.** The report gives no count for "nearly all". We read it as at least 95% of the 461, but fewer than all of them, having both fields. All 461 have a Trouble Code, so the 458 with a Category (99.3%) have both. The counts that the report's words rest on are in docs/04: 461 of 461 with a Trouble Code (§1 and §5) and 99.3% with a Category (§5). The report does not print them, so we check them in this table, apart from the check against the report.

In [9]:
NEARLY_ALL = 0.95  # our reading of "nearly all": at least 95%, but fewer than all


def how_many(count, total):
    """The report's words for how many of a group have both fields: 'all', 'nearly all', or the count."""
    if count == total:
        return 'all'
    if count >= NEARLY_ALL * total:
        return 'nearly all'
    return f'{count} of {total}'


aug_both_words = how_many(aug_both, n_aug_closed)
docs04 = pd.DataFrame([
    ('Of the 461, with a Trouble Code (docs/04 §1 and §5)', '461 of 461', f'{aug_code} of {n_aug_closed}'),
    ('Of the 461, with a Category (docs/04 §5)', '99.3%', R.pct(aug_category_share, 1)),
], columns=['Finding', 'docs/04', 'Recomputed'])
docs04['Match'] = (docs04['docs/04'] == docs04['Recomputed']).map({True: 'yes', False: 'NO'})
print(f'Of the {n_aug_closed}, {aug_both} have both fields: {aug_both_words}.')
assert (docs04.Match == 'yes').all(), "a recomputed count differs from docs/04's"
docs04

Of the 461, 458 have both fields: nearly all.


,Finding,docs/04,Recomputed,Match
0,"Of the 461, with a Trouble Code (docs/04 §1 and §5)",461 of 461,461 of 461,yes
1,"Of the 461, with a Category (docs/04 §5)",99.3%,99.3%,yes


## Category set near close, and final values only

IPA's answers at the Oct 13th meeting, which docs/04 records in its update of Oct 14th and in question 3 of §6, were that Trouble Code is entered at intake and may be updated by close, so the export holds only its final value, and that Category and Sub Category are set near the end of a ticket's life. These are records of IPA's answers, so the check has no row for them. The counts above fit Category being set near close: both fields are on 1 of the 64 open tickets vs. nearly all of the same month's Trouble Closed tickets. For Trouble Code, which IPA said is entered at intake, 63 of the 64 open tickets have none in the export, so docs/04 reads the export's field as filled by close and lists where the intake value is kept as an open question for IPA (question 3 in §6).

## Check against the report

The check leaves out IPA's answers, which are records of a meeting. The report gives no count for "nearly all", so that row applies our reading of the words: at least 95% of the 461, but fewer than all of them, have both fields.

In [10]:
R.check([
    ('Steps between consecutive tickets in case-number order', '7,470', R.num(n_steps)),
    ('Backward steps on the export clock', '0', R.num(export.backward)),
    ('Backward steps on hospital-local clocks, over the two zone maps', '1,295–1,312',
     R.span(shown.backward.min(), shown.backward.max())),
    ('Open tickets (status Dispatch)', '64', R.num(n_open)),
    ('Open tickets with a Trouble Code', '1', R.num(open_code)),
    ('Open tickets with a Category', '1', R.num(open_category)),
    ('Month compared (all open tickets were opened in it)', 'August 2026', month_label),
    ('Tickets opened that month with status Trouble Closed', '461', R.num(n_aug_closed)),
    ('Of the 461, with both a Trouble Code and a Category', 'nearly all', aug_both_words),
]);

,Finding,Report,Recomputed,Match
0,Steps between consecutive tickets in case-number order,"7,470","7,470",yes
1,Backward steps on the export clock,0,0,yes
2,"Backward steps on hospital-local clocks, over the two zone maps","1,295–1,312","1,295–1,312",yes
3,Open tickets (status Dispatch),64,64,yes
4,Open tickets with a Trouble Code,1,1,yes
5,Open tickets with a Category,1,1,yes
6,Month compared (all open tickets were opened in it),August 2026,August 2026,yes
7,Tickets opened that month with status Trouble Closed,461,461,yes
8,"Of the 461, with both a Trouble Code and a Category",nearly all,nearly all,yes


All 9 values match the report.
